# Important: Run this code cell each time you start a new session!

In [6]:
import os, datetime, json, locale, pathlib, urllib, requests, werkzeug, nbformat, google, yaml, warnings
def colab2pdf():
    locale.setlocale(locale.LC_ALL, 'en_US.UTF-8')
    NAME = pathlib.Path(werkzeug.utils.secure_filename(urllib.parse.unquote(requests.get(f"http://{os.environ['COLAB_JUPYTER_IP']}:{os.environ['KMP_TARGET_PORT']}/api/sessions").json()[0]["name"])))
    TEMP = pathlib.Path("/content/pdfs") / f"{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_{NAME.stem}"; TEMP.mkdir(parents=True, exist_ok=True)
    NB = [cell for cell in nbformat.reads(json.dumps(google.colab._message.blocking_request("get_ipynb", timeout_sec=30)["ipynb"]), as_version=4).cells if "--Colab2PDF" not in cell.source]
    warnings.filterwarnings('ignore', category=nbformat.validator.MissingIDFieldWarning)
    with (TEMP / f"{NAME.stem}.ipynb").open("w", encoding="utf-8") as nb_copy: nbformat.write(nbformat.v4.new_notebook(cells=NB or [nbformat.v4.new_code_cell("#")]), nb_copy)
    if not pathlib.Path("/usr/local/bin/quarto").exists():
        !wget -q "https://quarto.org/download/latest/quarto-linux-amd64.deb" -P {TEMP} && dpkg -i {TEMP}/quarto-linux-amd64.deb > /dev/null && quarto install tinytex --update-path --quiet
    with (TEMP / "config.yml").open("w", encoding="utf-8") as file: yaml.dump({'include-in-header': [{"text": r"\usepackage{fvextra}\DefineVerbatimEnvironment{Highlighting}{Verbatim}{breaksymbolleft={},showspaces=false,showtabs=false,breaklines,breakanywhere,commandchars=\\\{\}}"}],'include-before-body': [{"text": r"\DefineVerbatimEnvironment{verbatim}{Verbatim}{breaksymbolleft={},showspaces=false,showtabs=false,breaklines}"}]}, file)
    !quarto render {TEMP}/{NAME.stem}.ipynb --metadata-file={TEMP}/config.yml --to pdf -M latex-auto-install -M margin-top=1in -M margin-bottom=1in -M margin-left=1in -M margin-right=1in --quiet
    google.colab.files.download(str(TEMP / f"{NAME.stem}.pdf"))

# Assignment 5 Code



In this project, we explored methods of quantifying the skills and abilities of Formula 1 drivers based on historical data. We defined metrics for drivers and calculates an overall driver score using a random forest algorithm. Then, we isolated and calculated the impact of driver abilities, team and car performance, and circuit differences. This allowed us to compute a lap time distribution for every combination of driver, car, and circuit.

Using the computations, we built a simulation program for a hypothetical race. After the user chooses the drivers, cars, and circuit, we simulate exact lap times from the distributions and predicts the winner.

## Part 0: Configuration

Import necessary libraries and load dataset.

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, Tuple, Optional
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
import ipywidgets as widgets
from IPython.display import display, clear_output
import kagglehub

# Download dataset from KaggleHub
DATA_DIR = kagglehub.dataset_download("rohanrao/formula-1-world-championship-1950-2020")
print("Dataset path:", DATA_DIR)
MIN_QUAL_SAMPLES = 4
MIN_RACE_LAPS_FOR_STABILITY = 8
DEFAULT_SIM_LAPS = 50
RANDOM_SEED = 10
np.random.seed(RANDOM_SEED)

Using Colab cache for faster access to the 'formula-1-world-championship-1950-2020' dataset.
Dataset path: /kaggle/input/formula-1-world-championship-1950-2020


## Part 1: Loading / cleaning helpers

Helper functions for data loading and cleaning.

In [8]:
def load_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(f"{DATA_DIR}/{name}")

# Parse the lap time to milliseconds
def parse_lap_time_to_ms(x):
    if pd.isna(x) or x == r"\N":
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip()
    if not s:
        return np.nan
    parts = s.split(':')
    try:
        if len(parts) == 2:
            m = int(parts[0])
            sec = float(parts[1])
            return (m * 60 + sec) * 1000
        elif len(parts) == 3:
            h = int(parts[0])
            m = int(parts[1])
            sec = float(parts[2])
            return (h * 3600 + m * 60 + sec) * 1000
    except:
        return np.nan
    return np.nan


def best_qualifying_ms(row):
    vals = [parse_lap_time_to_ms(row[c]) for c in ["q1", "q2", "q3"] if c in row.index]
    vals = [v for v in vals if pd.notna(v)]
    return min(vals) if vals else np.nan


def robust_z(x: pd.Series) -> pd.Series:
    mu = x.mean()
    sigma = x.std(ddof=0)
    if pd.isna(sigma) or sigma == 0:
        return pd.Series(np.zeros(len(x)), index=x.index)
    return (x - mu) / sigma

## Part 2: Read core tables

Load data from files and define master dimensions.

In [9]:
# Load data from csv files
races = load_csv("races.csv")
results = load_csv("results.csv")
qualifying = load_csv("qualifying.csv")
lap_times = load_csv("lap_times.csv")
pit_stops = load_csv("pit_stops.csv")
drivers = load_csv("drivers.csv")
constructors = load_csv("constructors.csv")
circuits = load_csv("circuits.csv")
status = load_csv("status.csv")

# Master dimensions
race_dim = races[["raceId", "year", "round", "circuitId", "name", "date"]].copy()
race_dim = race_dim.rename(columns={"name": "raceName"})

driver_dim = drivers[["driverId", "code", "forename", "surname"]].copy()
driver_dim["driverName"] = (driver_dim["forename"].fillna("") + " " + driver_dim["surname"].fillna("")).str.strip()
constructor_dim = constructors[["constructorId", "name"]].rename(columns={"name": "constructorName"})
circuit_dim = circuits[["circuitId", "name"]].rename(columns={"name": "circuitName"})

## Part 3: Driver Evaluation

Calculates a driver score using three metrics:
1. Best Qualifying lap time
2. Race pace stability
3. Wheel-to-wheel ability

These metrics are compared with the field and combined into an overall driver score.

In [10]:
# Driver metric 1: qualifying lap time

qual = qualifying.copy()
qual["bestQualMs"] = qual.apply(best_qualifying_ms, axis=1)
qual = qual.merge(race_dim, on="raceId", how="left")
qual = qual.merge(driver_dim[["driverId", "driverName", "code"]], on="driverId", how="left")
qual = qual.merge(constructor_dim, on="constructorId", how="left")
qual = qual.dropna(subset=["bestQualMs"])

# Compare each driver's best qualifying lap time against field average
# Express lower lap time as a positive difference
qual_field = qual.groupby("raceId")["bestQualMs"].agg(field_mean="mean", field_std=lambda s: s.std(ddof=0)).reset_index()
qual = qual.merge(qual_field, on="raceId", how="left")
qual["qual_delta_ms"] = qual["bestQualMs"] - qual["field_mean"]
qual["qual_z"] = -(qual["qual_delta_ms"] / qual["field_std"].replace(0, np.nan))
qual["qual_z"] = qual["qual_z"].replace([np.inf, -np.inf], np.nan)

# Aggregated driver qualifying score by year
qual_driver_year = (
    qual.groupby(["driverId", "driverName", "year"])
    .agg(qual_score=("qual_z", "mean"), qual_events=("raceId", "nunique"))
    .reset_index()
)

# Aggregated career driver qualifying score for the driver
qual_driver_career = (
    qual.groupby(["driverId", "driverName"])
    .agg(driver_qual_score=("qual_z", "mean"), qual_events=("raceId", "nunique"))
    .reset_index()
)

In [11]:
# Driver metric 2: race pace stability
# Difference between best and worst lap time (excluding pit stop laps)

laps = lap_times.merge(race_dim, on="raceId", how="left")
laps = laps.merge(driver_dim[["driverId", "driverName", "code"]], on="driverId", how="left")

# Remove outliers and first lap
laps = laps[laps["lap"] > 1].copy()
laps = laps[laps["milliseconds"].notna()].copy()

race_median = laps.groupby("raceId")["milliseconds"].median().rename("race_median_ms")
race_iqr = laps.groupby("raceId")["milliseconds"].quantile([0.25, 0.75]).unstack()
race_iqr.columns = ["q25", "q75"]
race_iqr["iqr"] = race_iqr["q75"] - race_iqr["q25"]
clean_ref = pd.concat([race_median, race_iqr], axis=1).reset_index()
laps = laps.merge(clean_ref, on="raceId", how="left")
laps = laps[(laps["milliseconds"] >= laps["q25"] - 1.5 * laps["iqr"]) &
            (laps["milliseconds"] <= laps["q75"] + 1.5 * laps["iqr"])].copy()

race_stability = (
    laps.groupby(["raceId", "driverId", "driverName", "year", "circuitId"])
    .agg(
        lap_count=("lap", "count"),
        race_mean_ms=("milliseconds", "mean"),
        race_std_ms=("milliseconds", "std"),
        race_min_ms=("milliseconds", "min"),
        race_max_ms=("milliseconds", "max"),
    )
    .reset_index()
)
race_stability = race_stability[race_stability["lap_count"] >= MIN_RACE_LAPS_FOR_STABILITY].copy()
race_stability["race_range_ms"] = race_stability["race_max_ms"] - race_stability["race_min_ms"]

# Compare each driver's std and range with the field for the race
field_stability = (
    race_stability.groupby("raceId")
    .agg(
        field_std_mean=("race_std_ms", "mean"),
        field_std_std=("race_std_ms", lambda s: s.std(ddof=0)),
        field_range_mean=("race_range_ms", "mean"),
        field_range_std=("race_range_ms", lambda s: s.std(ddof=0)),
    )
    .reset_index()
)
race_stability = race_stability.merge(field_stability, on="raceId", how="left")

# Lower variability is better, so flip signs
race_stability["consistency_std_z"] = -(
    (race_stability["race_std_ms"] - race_stability["field_std_mean"]) /
    race_stability["field_std_std"].replace(0, np.nan)
)
race_stability["consistency_range_z"] = -(
    (race_stability["race_range_ms"] - race_stability["field_range_mean"]) /
    race_stability["field_range_std"].replace(0, np.nan)
)
race_stability["consistency_score_race"] = race_stability[["consistency_std_z", "consistency_range_z"]].mean(axis=1)

consistency_driver_career = (
    race_stability.groupby(["driverId", "driverName"])
    .agg(
        driver_consistency_score=("consistency_score_race", "mean"),
        consistency_races=("raceId", "nunique")
    )
    .reset_index()
)

In [12]:
# Driver metric 3: wheel-to-wheel abilities
# Calculate the difference between start and finish position

res = results.merge(race_dim, on="raceId", how="left")
res = res.merge(driver_dim[["driverId", "driverName", "code"]], on="driverId", how="left")
res = res.merge(constructor_dim, on="constructorId", how="left")
res["grid_clean"] = res["grid"].replace(0, np.nan)
res["finish_clean"] = res["positionOrder"]
res["positions_gained"] = res["grid_clean"] - res["finish_clean"]

field_pos = (
    res.groupby("raceId")["positions_gained"]
    .agg(field_gain_mean="mean", field_gain_std=lambda s: s.std(ddof=0))
    .reset_index()
)
res = res.merge(field_pos, on="raceId", how="left")
res["racecraft_z"] = (
    (res["positions_gained"] - res["field_gain_mean"]) /
    res["field_gain_std"].replace(0, np.nan)
)

racecraft_driver_career = (
    res.groupby(["driverId", "driverName"])
    .agg(driver_racecraft_score=("racecraft_z", "mean"), racecraft_races=("raceId", "nunique"))
    .reset_index()
)

In [13]:
# Overall driver score

driver_features = qual_driver_career.merge(consistency_driver_career, on=["driverId", "driverName"], how="outer")
driver_features = driver_features.merge(racecraft_driver_career, on=["driverId", "driverName"], how="outer")

points_target = (
    res.groupby(["driverId", "driverName"])
    .agg(avg_points_per_race=("points", "mean"), races=("raceId", "nunique"))
    .reset_index()
)
driver_model_df = driver_features.merge(points_target, on=["driverId", "driverName"], how="inner")

feature_cols_driver = [
    "driver_qual_score",
    "driver_consistency_score",
    "driver_racecraft_score",
]
for c in feature_cols_driver:
    driver_model_df[c] = driver_model_df[c].fillna(driver_model_df[c].mean())

X_driver = driver_model_df[feature_cols_driver]
y_driver = driver_model_df["avg_points_per_race"]

# Compute metric importance using random forest
driver_rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=3,
    random_state=RANDOM_SEED,
)
driver_rf.fit(X_driver, y_driver)

driver_model_df["driver_pred_points"] = driver_rf.predict(X_driver)
driver_model_df["driver_score_100"] = 100 * (
    (driver_model_df["driver_pred_points"] - driver_model_df["driver_pred_points"].min()) /
    (driver_model_df["driver_pred_points"].max() - driver_model_df["driver_pred_points"].min() + 1e-9)
)

# Raw random forest feature importance
raw_driver_importance = pd.DataFrame({
    "feature": feature_cols_driver,
    "importance": driver_rf.feature_importances_
})
raw_driver_importance["weight_pct"] = 100 * raw_driver_importance["importance"] / raw_driver_importance["importance"].sum()

# Permutation importance
driver_perm = permutation_importance(
    driver_rf, X_driver, y_driver, n_repeats=20, random_state=RANDOM_SEED
)
driver_importance = pd.DataFrame({
    "feature": feature_cols_driver,
    "importance_mean": driver_perm.importances_mean,
    "importance_std": driver_perm.importances_std,
})
driver_importance["weight_pct"] = 100 * driver_importance["importance_mean"] / driver_importance["importance_mean"].sum()

print("Driver RF raw feature importances:")
print(raw_driver_importance.sort_values("importance", ascending=False))
print("Driver RF permutation importances:")
print(driver_importance.sort_values("importance_mean", ascending=False))

Driver RF raw feature importances:
                    feature  importance  weight_pct
0         driver_qual_score    0.710315   71.031539
2    driver_racecraft_score    0.272408   27.240796
1  driver_consistency_score    0.017277    1.727666
Driver RF permutation importances:
                    feature  importance_mean  importance_std  weight_pct
0         driver_qual_score         1.591203        0.114357   76.919939
2    driver_racecraft_score         0.465476        0.059207   22.501446
1  driver_consistency_score         0.011970        0.001036    0.578616


## Part 4: Constructor Evaluation

Computes a team score using three metrics:
1. Average finishing position
2. Average lap time
3. Pit-stop performance
These metrics are compared with the field.

In [14]:
# Team score

# Finishing position
res_team = results.merge(race_dim, on="raceId", how="left")
res_team = res_team.merge(constructor_dim, on="constructorId", how="left")

team_finish = (
    res_team.groupby(["year", "constructorId", "constructorName"])
    .agg(avg_finish_pos=("positionOrder", "mean"), races=("raceId", "nunique"))
    .reset_index()
)
team_finish["finish_z_within_year"] = team_finish.groupby("year")["avg_finish_pos"].transform(
    lambda s: -(s - s.mean()) / (s.std(ddof=0) if s.std(ddof=0) != 0 else 1)
)

# Lap pace
laps_team = laps.merge(res_team[["raceId", "driverId", "constructorId", "constructorName"]], on=["raceId", "driverId"], how="left")
team_lap_pace = (
    laps_team.groupby(["year", "constructorId", "constructorName"])
    .agg(avg_lap_ms=("milliseconds", "mean"), lap_count=("lap", "count"))
    .reset_index()
)
team_lap_pace["lap_pace_z_within_year"] = team_lap_pace.groupby("year")["avg_lap_ms"].transform(
    lambda s: -(s - s.mean()) / (s.std(ddof=0) if s.std(ddof=0) != 0 else 1)
)

In [15]:
# Pit stop
pit = pit_stops.merge(race_dim, on="raceId", how="left")
pit = pit.merge(res_team[["raceId", "driverId", "constructorId", "constructorName"]].drop_duplicates(), on=["raceId", "driverId"], how="left")
pit = pit[pit["milliseconds"].notna()].copy()

team_pit = (
    pit.groupby(["year", "constructorId", "constructorName"])
    .agg(
        avg_pit_ms=("milliseconds", "mean"),
        pit_std_ms=("milliseconds", "std"),
        pit_min_ms=("milliseconds", "min"),
        pit_max_ms=("milliseconds", "max"),
        pit_count=("milliseconds", "count"),
    )
    .reset_index()
)
team_pit["pit_range_ms"] = team_pit["pit_max_ms"] - team_pit["pit_min_ms"]
team_pit["pit_avg_z_within_year"] = team_pit.groupby("year")["avg_pit_ms"].transform(
    lambda s: -(s - s.mean()) / (s.std(ddof=0) if s.std(ddof=0) != 0 else 1)
)
team_pit["pit_consistency_z_within_year"] = team_pit.groupby("year")["pit_std_ms"].transform(
    lambda s: -(s - s.mean()) / (s.std(ddof=0) if s.std(ddof=0) != 0 else 1)
)
team_pit["pit_range_z_within_year"] = team_pit.groupby("year")["pit_range_ms"].transform(
    lambda s: -(s - s.mean()) / (s.std(ddof=0) if s.std(ddof=0) != 0 else 1)
)
team_pit["pit_overall_z_within_year"] = team_pit[["pit_avg_z_within_year", "pit_consistency_z_within_year", "pit_range_z_within_year"]].mean(axis=1)

In [16]:
# Final team score

team_features = team_finish.merge(team_lap_pace, on=["year", "constructorId", "constructorName"], how="outer")
team_features = team_features.merge(team_pit[["year", "constructorId", "constructorName", "pit_overall_z_within_year"]], on=["year", "constructorId", "constructorName"], how="left")

team_target = (
    res_team.groupby(["year", "constructorId", "constructorName"])
    .agg(avg_team_points=("points", "mean"), races=("raceId", "nunique"))
    .reset_index()
)
team_model_df = team_features.merge(team_target, on=["year", "constructorId", "constructorName"], how="inner")
feature_cols_team = [
    "finish_z_within_year",
    "lap_pace_z_within_year",
    "pit_overall_z_within_year",
]
for c in feature_cols_team:
    team_model_df[c] = team_model_df[c].fillna(team_model_df[c].mean())

X_team = team_model_df[feature_cols_team]
y_team = team_model_df["avg_team_points"]

team_rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=2,
    random_state=RANDOM_SEED,
)
team_rf.fit(X_team, y_team)

team_model_df["team_pred_points"] = team_rf.predict(X_team)
team_model_df["team_score_100"] = 100 * (
    (team_model_df["team_pred_points"] - team_model_df["team_pred_points"].min()) /
    (team_model_df["team_pred_points"].max() - team_model_df["team_pred_points"].min() + 1e-9)
)

raw_team_importance = pd.DataFrame({
    "feature": feature_cols_team,
    "importance": team_rf.feature_importances_
})
raw_team_importance["weight_pct"] = 100 * raw_team_importance["importance"] / raw_team_importance["importance"].sum()

team_perm = permutation_importance(
    team_rf, X_team, y_team, n_repeats=20, random_state=RANDOM_SEED
)
team_importance = pd.DataFrame({
    "feature": feature_cols_team,
    "importance_mean": team_perm.importances_mean,
    "importance_std": team_perm.importances_std,
})
team_importance["weight_pct"] = 100 * team_importance["importance_mean"] / team_importance["importance_mean"].sum()

print("Constructor RF raw feature importances:")
print(raw_team_importance.sort_values("importance", ascending=False))
print("Constructor RF permutation importances:")
print(team_importance.sort_values("importance_mean", ascending=False))

Constructor RF raw feature importances:
                     feature  importance  weight_pct
1     lap_pace_z_within_year    0.462801   46.280119
0       finish_z_within_year    0.341066   34.106551
2  pit_overall_z_within_year    0.196133   19.613330
Constructor RF permutation importances:
                     feature  importance_mean  importance_std  weight_pct
0       finish_z_within_year         0.870008        0.054227   55.804564
2  pit_overall_z_within_year         0.414744        0.036081   26.602720
1     lap_pace_z_within_year         0.274275        0.011390   17.592716


## Part 5: Circuit baseline distributions

For each circuit, a normal distribution for baseline lap time is calculated.

In [17]:
# Lap time distribution

circuit_baseline = (
    laps.groupby(["circuitId", "year"])
    .agg(
        circuit_mu_ms=("milliseconds", "mean"),
        circuit_sigma_ms=("milliseconds", "std"),
        lap_count=("lap", "count")
    )
    .reset_index()
    .merge(circuit_dim, on="circuitId", how="left")
)

circuit_baseline_all_years = (
    laps.groupby(["circuitId"])
    .agg(
        circuit_mu_ms=("milliseconds", "mean"),
        circuit_sigma_ms=("milliseconds", "std"),
        lap_count=("lap", "count")
    )
    .reset_index()
    .merge(circuit_dim, on="circuitId", how="left")
)

## Part 6: Driver-car-circuit lap distribution

Using the driver score, team score, and circuit baseline calculated in previous parts, a lap time distribution (mean and standard deviation) is calculated for all possible combinations of driver, car and circuit.

Distribution:
- Lap_time ~ Normal(mu, sigma)
- mu = circuit baseline + driver effect + team effect
- sigma = adjusted from circuit sigma and driver+team+era effects

In [18]:
def normalize_score(series: pd.Series) -> pd.Series:
    smin, smax = series.min(), series.max()
    return (series - smin) / (smax - smin + 1e-9)

# Standardize driver and team scores
driver_score_map = driver_model_df[["driverId", "driverName", "driver_score_100", "driver_qual_score", "driver_consistency_score", "driver_racecraft_score"]].copy()
team_score_map = team_model_df[["year", "constructorId", "constructorName", "team_score_100"]].copy()

DRIVER_PACE_MS_PER_Z = -180.0
TEAM_PACE_MS_PER_SCORE100 = -8.0
SIGMA_CONSISTENCY_MS_PER_Z = -40.0
MIN_SIGMA_MS = 80.0

# Calculates driver-car-circuit lap distribution
def get_driver_distribution(driver_id: int, constructor_id: int, year: int, circuit_id: int) -> Dict[str, float]:
    cb = circuit_baseline[(circuit_baseline["circuitId"] == circuit_id) & (circuit_baseline["year"] == year)]
    if cb.empty:
        cb = circuit_baseline_all_years[circuit_baseline_all_years["circuitId"] == circuit_id]
    if cb.empty:
        raise ValueError("No circuit baseline available for this circuit")
    cb = cb.iloc[0]

    drow = driver_model_df[driver_model_df["driverId"] == driver_id]
    if drow.empty:
        raise ValueError("Driver not found in driver model table")
    drow = drow.iloc[0]

    trow = team_model_df[(team_model_df["year"] == year) & (team_model_df["constructorId"] == constructor_id)]
    if trow.empty:
        trow = team_model_df[team_model_df["constructorId"] == constructor_id]
    if trow.empty:
        raise ValueError("Constructor not found in team model table")
    trow = trow.iloc[-1]

    mu = cb["circuit_mu_ms"]
    sigma = cb["circuit_sigma_ms"] if pd.notna(cb["circuit_sigma_ms"]) else 300.0

    if pd.notna(drow.get("driver_qual_score", np.nan)):
        mu += DRIVER_PACE_MS_PER_Z * drow["driver_qual_score"]

    if pd.notna(trow.get("team_score_100", np.nan)):
        mu += TEAM_PACE_MS_PER_SCORE100 * (trow["team_score_100"] - 50) / 10.0

    if pd.notna(drow.get("driver_consistency_score", np.nan)):
        sigma += SIGMA_CONSISTENCY_MS_PER_Z * drow["driver_consistency_score"]

    sigma = max(MIN_SIGMA_MS, sigma)

    return {
        "driverId": driver_id,
        "constructorId": constructor_id,
        "year": year,
        "circuitId": circuit_id,
        "mu_ms": float(mu),
        "sigma_ms": float(sigma),
    }

## Part 7: Monte Carlo simulator for two driver-car combinations

This section contains functions used for running the simulation for a race between two chosen drivers, cars on any circuit.

In [19]:
# Simulation parameters
@dataclass
class SimSpec:
    driver_id: int
    constructor_id: int
    year: int
    circuit_id: int
    label: str

# If the user does not specify the number of laps, use the default
# number of laps for the circuit
def get_default_race_laps(circuit_id: int, preferred_year: Optional[int] = None) -> Optional[int]:
    race_subset = race_dim[race_dim["circuitId"] == circuit_id].copy()
    if race_subset.empty:
        return None

    lap_counts = (
        lap_times.merge(race_dim[["raceId", "year", "circuitId"]], on="raceId", how="left")
        .groupby("raceId")["lap"].max()
        .reset_index()
        .merge(race_dim[["raceId", "year", "circuitId"]], on="raceId", how="left")
    )
    lap_counts = lap_counts[lap_counts["circuitId"] == circuit_id].copy()
    if lap_counts.empty:
        return None

    if preferred_year is not None:
        exact = lap_counts[lap_counts["year"] == preferred_year]
        if not exact.empty:
            return int(round(exact["lap"].median()))

    return int(round(lap_counts["lap"].median()))

# Monte Carlo Simulator
def simulate_head_to_head(spec_a: SimSpec, spec_b: SimSpec, n_laps: Optional[int] = None, n_sims: int = 2000):
    dist_a = get_driver_distribution(spec_a.driver_id, spec_a.constructor_id, spec_a.year, spec_a.circuit_id)
    dist_b = get_driver_distribution(spec_b.driver_id, spec_b.constructor_id, spec_b.year, spec_b.circuit_id)

    if n_laps is None:
        n_laps = get_default_race_laps(spec_a.circuit_id, preferred_year=spec_a.year)
    if n_laps is None:
        n_laps = get_default_race_laps(spec_b.circuit_id, preferred_year=spec_b.year)
    if n_laps is None:
        n_laps = DEFAULT_SIM_LAPS

    laps_a = np.random.normal(dist_a["mu_ms"], dist_a["sigma_ms"], size=(n_sims, n_laps))
    laps_b = np.random.normal(dist_b["mu_ms"], dist_b["sigma_ms"], size=(n_sims, n_laps))

    cum_a = np.cumsum(laps_a, axis=1) / 1000.0
    cum_b = np.cumsum(laps_b, axis=1) / 1000.0
    cum_gap = cum_a - cum_b

    mean_cum_a = cum_a.mean(axis=0)
    mean_cum_b = cum_b.mean(axis=0)
    mean_gap = cum_gap.mean(axis=0)
    p10_gap = np.percentile(cum_gap, 10, axis=0)
    p90_gap = np.percentile(cum_gap, 90, axis=0)

    p10_cum_a = np.percentile(cum_a, 10, axis=0)
    p90_cum_a = np.percentile(cum_a, 90, axis=0)
    p10_cum_b = np.percentile(cum_b, 10, axis=0)
    p90_cum_b = np.percentile(cum_b, 90, axis=0)

    cb = circuit_baseline[(circuit_baseline["circuitId"] == spec_a.circuit_id) & (circuit_baseline["year"] == spec_a.year)]
    if cb.empty:
        cb = circuit_baseline_all_years[circuit_baseline_all_years["circuitId"] == spec_a.circuit_id]
    circuit_baseline_sec = float(cb.iloc[0]["circuit_mu_ms"] / 1000.0) if not cb.empty else np.nan
    circuit_baseline_cumulative_sec = np.arange(1, n_laps + 1) * circuit_baseline_sec if pd.notna(circuit_baseline_sec) else np.repeat(np.nan, n_laps)

    driver_a_vs_baseline_sec = mean_cum_a - circuit_baseline_cumulative_sec
    driver_b_vs_baseline_sec = mean_cum_b - circuit_baseline_cumulative_sec
    driver_a_p10_vs_baseline_sec = p10_cum_a - circuit_baseline_cumulative_sec
    driver_a_p90_vs_baseline_sec = p90_cum_a - circuit_baseline_cumulative_sec
    driver_b_p10_vs_baseline_sec = p10_cum_b - circuit_baseline_cumulative_sec
    driver_b_p90_vs_baseline_sec = p90_cum_b - circuit_baseline_cumulative_sec

    final_gap = cum_gap[:, -1]
    a_wins = int(np.sum(final_gap < 0))
    b_wins = int(np.sum(final_gap > 0))
    ties = int(np.sum(np.isclose(final_gap, 0, atol=1e-9)))
    a_win_pct = 100.0 * a_wins / n_sims
    b_win_pct = 100.0 * b_wins / n_sims
    tie_pct = 100.0 * ties / n_sims

    out = pd.DataFrame({
        "lap": np.arange(1, n_laps + 1),
        "driver_a_cum_sec": mean_cum_a,
        "driver_b_cum_sec": mean_cum_b,
        "driver_a_vs_baseline_sec": driver_a_vs_baseline_sec,
        "driver_b_vs_baseline_sec": driver_b_vs_baseline_sec,
        "driver_a_p10_vs_baseline_sec": driver_a_p10_vs_baseline_sec,
        "driver_a_p90_vs_baseline_sec": driver_a_p90_vs_baseline_sec,
        "driver_b_p10_vs_baseline_sec": driver_b_p10_vs_baseline_sec,
        "driver_b_p90_vs_baseline_sec": driver_b_p90_vs_baseline_sec,
        "mean_gap_sec": mean_gap,
        "p10_gap_sec": p10_gap,
        "p90_gap_sec": p90_gap,
        "circuit_baseline_cumulative_sec": circuit_baseline_cumulative_sec,
        "zero_baseline": np.zeros(n_laps),
        "label_a": spec_a.label,
        "label_b": spec_b.label,
    })

    summary = {
        "n_sims": int(n_sims),
        "a_wins": a_wins,
        "b_wins": b_wins,
        "ties": ties,
        "a_win_pct": a_win_pct,
        "b_win_pct": b_win_pct,
        "tie_pct": tie_pct,
        "final_mean_gap_sec": float(mean_gap[-1]),
        "final_p10_gap_sec": float(p10_gap[-1]),
        "final_p90_gap_sec": float(p90_gap[-1]),
    }
    return out, dist_a, dist_b, summary

# Plot the visualization for the result (gap between the two drivers)
def plot_gap(df_gap: pd.DataFrame, summary: Optional[dict] = None):
    plt.figure(figsize=(12, 6))

    plt.plot(df_gap["lap"], df_gap["driver_a_vs_baseline_sec"], label=df_gap["label_a"].iloc[0])
    plt.fill_between(
        df_gap["lap"],
        df_gap["driver_a_p10_vs_baseline_sec"],
        df_gap["driver_a_p90_vs_baseline_sec"],
        alpha=0.20
    )

    plt.plot(df_gap["lap"], df_gap["driver_b_vs_baseline_sec"], label=df_gap["label_b"].iloc[0])
    plt.fill_between(
        df_gap["lap"],
        df_gap["driver_b_p10_vs_baseline_sec"],
        df_gap["driver_b_p90_vs_baseline_sec"],
        alpha=0.20
    )

    plt.axhline(0, linestyle="--", label="Circuit average baseline (0)")
    plt.xlabel("Lap")
    plt.ylabel("Cumulative deviation from circuit average (seconds)")
    plt.title(f"{df_gap['label_a'].iloc[0]} vs {df_gap['label_b'].iloc[0]}")
    plt.legend()

    if summary is not None:
        text = (
            f"{df_gap['label_a'].iloc[0]} wins: {summary['a_wins']} ({summary['a_win_pct']:.1f}%)\n"
            f"{df_gap['label_b'].iloc[0]} wins: {summary['b_wins']} ({summary['b_win_pct']:.1f}%)\n"
            f"Ties: {summary['ties']} ({summary['tie_pct']:.1f}%)\n"
            f"Final mean gap (A-B): {summary['final_mean_gap_sec']:.3f}s"
        )
        plt.gca().text(
            0.02, 0.98, text,
            transform=plt.gca().transAxes,
            verticalalignment='top',
            bbox=dict(boxstyle='round', alpha=0.15)
        )

    plt.tight_layout()
    plt.show()

## Part 8: Simulation helpers

Helper functions for looking up parameters for simulation.

In [20]:
def find_driver_id(name_fragment: str) -> pd.DataFrame:
    name_fragment = name_fragment.lower()
    out = driver_dim[driver_dim["driverName"].str.lower().str.contains(name_fragment, na=False)].copy()
    return out.sort_values("driverName")


def find_constructor_id(name_fragment: str) -> pd.DataFrame:
    name_fragment = name_fragment.lower()
    out = constructor_dim[constructor_dim["constructorName"].str.lower().str.contains(name_fragment, na=False)].copy()
    return out.sort_values("constructorName")


def find_circuit_id(name_fragment: str) -> pd.DataFrame:
    name_fragment = name_fragment.lower()
    out = circuit_dim[circuit_dim["circuitName"].str.lower().str.contains(name_fragment, na=False)].copy()
    return out.sort_values("circuitName")

Save computation results.

In [21]:
driver_model_df.to_csv("driver_scores.csv", index=False)
team_model_df.to_csv("constructor_scores.csv", index=False)
circuit_baseline.to_csv("circuit_baseline_by_year.csv", index=False)
circuit_baseline_all_years.to_csv("circuit_baseline_all_years.csv", index=False)
race_stability.to_csv("driver_race_stability_by_race.csv", index=False)
qual.to_csv("driver_qualifying_zscores_by_race.csv", index=False)
driver_importance.to_csv("driver_rf_importance.csv", index=False)
team_importance.to_csv("constructor_rf_importance.csv", index=False)

print("\nSaved:")
print("- driver_scores.csv")
print("- constructor_scores.csv")
print("- circuit_baseline_by_year.csv")
print("- circuit_baseline_all_years.csv")
print("- driver_race_stability_by_race.csv")
print("- driver_qualifying_zscores_by_race.csv")
print("- driver_rf_importance.csv")
print("- constructor_rf_importance.csv")


Saved:
- driver_scores.csv
- constructor_scores.csv
- circuit_baseline_by_year.csv
- circuit_baseline_all_years.csv
- driver_race_stability_by_race.csv
- driver_qualifying_zscores_by_race.csv
- driver_rf_importance.csv
- constructor_rf_importance.csv


## Part 9: Interactive simulation UI

Implements interactive UI for running simulation, allowing the user to easily choose the parameters: drivers, cars, circuit, laps (optional), and number of simulations. Then, outputs test and visual results.

In [22]:
# Driver options
driver_options = sorted([(f"{row.driverName} ({row.code if pd.notna(row.code) else row.driverId})", int(row.driverId))
                         for _, row in driver_dim.drop_duplicates("driverId").iterrows()])

# Car options (constructor + year)
car_year_df = (
    team_model_df[["year", "constructorId", "constructorName"]]
    .drop_duplicates()
    .sort_values(["constructorName", "year"])
    .copy()
)
car_year_df["carLabel"] = car_year_df["constructorName"] + " " + car_year_df["year"].astype(str)
car_options = [(row.carLabel, (int(row.constructorId), int(row.year))) for _, row in car_year_df.iterrows()]

circuit_options = sorted([(row.circuitName, int(row.circuitId))
                          for _, row in circuit_dim.drop_duplicates("circuitId").iterrows()])

# Use Combobox for convenient typing / searching functionalities
driver_a_box = widgets.Combobox(
    placeholder='Search driver A',
    options=[x[0] for x in driver_options],
    description='Driver A:',
    ensure_option=True,
    layout=widgets.Layout(width='450px')
)
driver_b_box = widgets.Combobox(
    placeholder='Search driver B',
    options=[x[0] for x in driver_options],
    description='Driver B:',
    ensure_option=True,
    layout=widgets.Layout(width='450px')
)
car_a_box = widgets.Combobox(
    placeholder='Search car A (constructor + year)',
    options=[x[0] for x in car_options],
    description='Car A:',
    ensure_option=True,
    layout=widgets.Layout(width='450px')
)
car_b_box = widgets.Combobox(
    placeholder='Search car B (constructor + year)',
    options=[x[0] for x in car_options],
    description='Car B:',
    ensure_option=True,
    layout=widgets.Layout(width='450px')
)
circuit_box = widgets.Combobox(
    placeholder='Search circuit',
    options=[x[0] for x in circuit_options],
    description='Circuit:',
    ensure_option=True,
    layout=widgets.Layout(width='450px')
)
manual_laps_checkbox = widgets.Checkbox(value=False, description='Override default laps')
laps_slider = widgets.IntSlider(value=50, min=5, max=100, step=1, description='Laps:')
laps_slider.layout.display = 'none'
sims_slider = widgets.IntSlider(value=3000, min=500, max=10000, step=500, description='Sims:')

def toggle_laps_visibility(change):
    laps_slider.layout.display = '' if change['new'] else 'none'

manual_laps_checkbox.observe(toggle_laps_visibility, names='value')
run_button = widgets.Button(description='Compare', button_style='success')
out = widgets.Output()


def _name_to_id(name, pairs):
    lookup = {k: v for k, v in pairs}
    if name not in lookup:
        raise ValueError(f"Selection not found: {name}")
    return lookup[name]


def compare_callback(_):
    with out:
        clear_output(wait=True)
        try:
            driver_a_id = _name_to_id(driver_a_box.value, driver_options)
            driver_b_id = _name_to_id(driver_b_box.value, driver_options)
            car_a = _name_to_id(car_a_box.value, car_options)
            car_b = _name_to_id(car_b_box.value, car_options)
            circuit_id = _name_to_id(circuit_box.value, circuit_options)

            constructor_a_id, year_a = car_a
            constructor_b_id, year_b = car_b

            spec_a = SimSpec(
                driver_id=driver_a_id,
                constructor_id=constructor_a_id,
                year=int(year_a),
                circuit_id=circuit_id,
                label=f"{driver_a_box.value} | {car_a_box.value}"
            )
            spec_b = SimSpec(
                driver_id=driver_b_id,
                constructor_id=constructor_b_id,
                year=int(year_b),
                circuit_id=circuit_id,
                label=f"{driver_b_box.value} | {car_b_box.value}"
            )

            selected_laps = int(laps_slider.value) if manual_laps_checkbox.value else None
            gap_df, dist_a, dist_b, sim_summary = simulate_head_to_head(
                spec_a, spec_b,
                n_laps=selected_laps,
                n_sims=int(sims_slider.value)
            )

            dist_a_pretty = {k: (round(v / 1000.0, 3) if k in ['mu_ms', 'sigma_ms'] else v) for k, v in dist_a.items()}
            dist_b_pretty = {k: (round(v / 1000.0, 3) if k in ['mu_ms', 'sigma_ms'] else v) for k, v in dist_b.items()}
            dist_a_pretty['mu_sec'] = dist_a_pretty.pop('mu_ms')
            dist_a_pretty['sigma_sec'] = dist_a_pretty.pop('sigma_ms')
            dist_b_pretty['mu_sec'] = dist_b_pretty.pop('mu_ms')
            dist_b_pretty['sigma_sec'] = dist_b_pretty.pop('sigma_ms')

            print("Distribution A:", dist_a_pretty)
            print("Distribution B:", dist_b_pretty)
            print("Default laps used:", int(gap_df['lap'].max()))
            print("Simulation summary:", sim_summary)
            print("Driver importance (permutation):")
            print(driver_importance.sort_values("weight_pct", ascending=False))
            print("Constructor importance (permutation):")
            print(team_importance.sort_values("weight_pct", ascending=False))

            plot_gap(gap_df, sim_summary)
            display(gap_df.head(15))
        except Exception as e:
            print("Error:", e)

run_button.on_click(compare_callback)

ui = widgets.VBox([
    widgets.HTML("<h3>F1 Head-to-Head Simulator</h3>"),
    driver_a_box,
    car_a_box,
    widgets.HTML("<hr>"),
    driver_b_box,
    car_b_box,
    widgets.HTML("<hr>"),
    circuit_box,
    manual_laps_checkbox,
    laps_slider,
    sims_slider,
    run_button,
    out,
])

display(ui)

## Part 10: Accuracy test

Tests the accuracy of our algorithm by running the simulation using real-life parameters. For every race in seasons 2018-2022, a simulation is performed on all possible pairs of drivers (from every driver that finished the race). Accuracy is then computed by comparing the actual race results against simulation results for each trial.

In [23]:
import pandas as pd
import numpy as np
from itertools import combinations

# Pairwise accuracy test

START_YEAR = 2018
END_YEAR = 2022
EXCLUDE_NON_CLASSIFIED = True
USE_FINISHERS_ONLY = False

def build_season_pairwise_accuracy(
    test_year,
    exclude_non_classified=True,
    use_finishers_only=False
):

    season_races = race_dim[race_dim["year"] == test_year].copy()
    if season_races.empty:
        return None, None, None, None

    season_results = results.merge(
        race_dim[["raceId", "year", "circuitId", "raceName", "date"]],
        on="raceId",
        how="left"
    )
    season_results = season_results[season_results["year"] == test_year].copy()

    season_results = season_results.merge(
        driver_dim[["driverId", "driverName", "code"]],
        on="driverId",
        how="left"
    )

    season_results = season_results.merge(
        constructor_dim[["constructorId", "constructorName"]],
        on="constructorId",
        how="left"
    )

    if exclude_non_classified:
        season_results = season_results[season_results["positionOrder"].notna()].copy()

    if use_finishers_only and "statusId" in season_results.columns and "status" in status.columns:
        season_results = season_results.merge(
            status[["statusId", "status"]],
            on="statusId",
            how="left"
        )
        finish_keywords = ["Finished", "+1 Lap", "+2 Laps", "+3 Laps", "+4 Laps", "+5 Laps", "+6 Laps"]
        season_results = season_results[
            season_results["status"].astype(str).isin(finish_keywords)
        ].copy()

    pair_records = []

    for race_id, race_df in season_results.groupby("raceId"):
        race_df = race_df.copy()

        if len(race_df) < 2:
            continue

        race_info = race_df.iloc[0]
        circuit_id = int(race_info["circuitId"])
        race_name = race_info["raceName"]

        idx_list = list(race_df.index)

        for i, j in combinations(idx_list, 2):
            row_a = race_df.loc[i]
            row_b = race_df.loc[j]

            driver_a_id = int(row_a["driverId"])
            driver_b_id = int(row_b["driverId"])
            constructor_a_id = int(row_a["constructorId"])
            constructor_b_id = int(row_b["constructorId"])

            pos_a = row_a["positionOrder"]
            pos_b = row_b["positionOrder"]

            if pd.isna(pos_a) or pd.isna(pos_b) or pos_a == pos_b:
                continue

            try:
                dist_a = get_driver_distribution(
                    driver_id=driver_a_id,
                    constructor_id=constructor_a_id,
                    year=test_year,
                    circuit_id=circuit_id
                )
                dist_b = get_driver_distribution(
                    driver_id=driver_b_id,
                    constructor_id=constructor_b_id,
                    year=test_year,
                    circuit_id=circuit_id
                )
            except Exception:
                continue

            pred_a_better = dist_a["mu_ms"] < dist_b["mu_ms"]

            actual_a_better = pos_a < pos_b

            correct = int(pred_a_better == actual_a_better)

            pair_records.append({
                "year": test_year,
                "raceId": race_id,
                "raceName": race_name,
                "circuitId": circuit_id,

                "driverA_id": driver_a_id,
                "driverA_name": row_a["driverName"],
                "driverA_code": row_a["code"],
                "constructorA_id": constructor_a_id,
                "constructorA_name": row_a["constructorName"],
                "driverA_pos": pos_a,
                "driverA_mu_ms": dist_a["mu_ms"],

                "driverB_id": driver_b_id,
                "driverB_name": row_b["driverName"],
                "driverB_code": row_b["code"],
                "constructorB_id": constructor_b_id,
                "constructorB_name": row_b["constructorName"],
                "driverB_pos": pos_b,
                "driverB_mu_ms": dist_b["mu_ms"],

                "predicted_better": row_a["driverName"] if pred_a_better else row_b["driverName"],
                "actual_better": row_a["driverName"] if actual_a_better else row_b["driverName"],
                "correct": correct
            })

    pairwise_df = pd.DataFrame(pair_records)

    if pairwise_df.empty:
        return None, None, None, None

    summary_overall = pd.DataFrame([{
        "year": test_year,
        "num_races": pairwise_df["raceId"].nunique(),
        "num_pairs": len(pairwise_df),
        "num_correct": int(pairwise_df["correct"].sum()),
        "accuracy": pairwise_df["correct"].mean()
    }])

    summary_by_race = (
        pairwise_df.groupby(["year", "raceId", "raceName", "circuitId"])
        .agg(
            num_pairs=("correct", "size"),
            num_correct=("correct", "sum"),
            accuracy=("correct", "mean")
        )
        .reset_index()
        .sort_values(["year", "accuracy"], ascending=[True, False])
    )

    driver_side_a = pairwise_df[[
        "year", "driverA_id", "driverA_name", "correct"
    ]].rename(columns={
        "driverA_id": "driverId",
        "driverA_name": "driverName"
    })

    driver_side_b = pairwise_df[[
        "year", "driverB_id", "driverB_name", "correct"
    ]].rename(columns={
        "driverB_id": "driverId",
        "driverB_name": "driverName"
    })

    driver_pairwise_long = pd.concat([driver_side_a, driver_side_b], ignore_index=True)

    summary_by_driver = (
        driver_pairwise_long.groupby(["year", "driverId", "driverName"])
        .agg(
            num_pairs=("correct", "size"),
            num_correct=("correct", "sum"),
            accuracy=("correct", "mean")
        )
        .reset_index()
        .sort_values(["year", "accuracy", "num_pairs"], ascending=[True, False, False])
    )

    return pairwise_df, summary_overall, summary_by_race, summary_by_driver


def build_range_pairwise_accuracy(
    start_year,
    end_year,
    exclude_non_classified=True,
    use_finishers_only=False
):
    all_pairwise = []
    all_overall = []
    all_by_race = []
    all_by_driver = []

    for year in range(start_year, end_year + 1):
        pairwise_df, summary_overall, summary_by_race, summary_by_driver = build_season_pairwise_accuracy(
            test_year=year,
            exclude_non_classified=exclude_non_classified,
            use_finishers_only=use_finishers_only
        )

        if pairwise_df is None:
            print(f"Skipping {year}: no valid comparisons.")
            continue

        all_pairwise.append(pairwise_df)
        all_overall.append(summary_overall)
        all_by_race.append(summary_by_race)
        all_by_driver.append(summary_by_driver)

        print(
            f"{year}: races={int(summary_overall['num_races'].iloc[0])}, "
            f"pairs={int(summary_overall['num_pairs'].iloc[0])}, "
            f"accuracy={summary_overall['accuracy'].iloc[0]:.4f}"
        )

    if len(all_pairwise) == 0:
        raise ValueError("No valid seasons found in the requested range.")

    pairwise_all = pd.concat(all_pairwise, ignore_index=True)
    overall_by_year = pd.concat(all_overall, ignore_index=True)
    by_race_all = pd.concat(all_by_race, ignore_index=True)
    by_driver_all = pd.concat(all_by_driver, ignore_index=True)

    overall_range = pd.DataFrame([{
        "start_year": start_year,
        "end_year": end_year,
        "num_years": overall_by_year["year"].nunique(),
        "num_races": pairwise_all["raceId"].nunique(),
        "num_pairs": len(pairwise_all),
        "num_correct": int(pairwise_all["correct"].sum()),
        "accuracy": pairwise_all["correct"].mean()
    }])

    return pairwise_all, overall_by_year, overall_range, by_race_all, by_driver_all

# Run tests for years specified
pairwise_all, overall_by_year, overall_range, by_race_all, by_driver_all = build_range_pairwise_accuracy(
    start_year=START_YEAR,
    end_year=END_YEAR,
    exclude_non_classified=EXCLUDE_NON_CLASSIFIED,
    use_finishers_only=USE_FINISHERS_ONLY
)

print("\n=== Overall Accuracy by Year ===")
print(overall_by_year)

print("\n=== Overall Accuracy for Full Range ===")
print(overall_range)

print("\n=== Best Races by Accuracy ===")
print(by_race_all.sort_values("accuracy", ascending=False).head(20))

print("\n=== Best Drivers by Accuracy ===")
print(by_driver_all.sort_values(["accuracy", "num_pairs"], ascending=[False, False]).head(20))

# Save output
pairwise_all.to_csv(f"pairwise_predictions_{START_YEAR}_{END_YEAR}.csv", index=False)
overall_by_year.to_csv(f"pairwise_accuracy_by_year_{START_YEAR}_{END_YEAR}.csv", index=False)
overall_range.to_csv(f"pairwise_accuracy_range_{START_YEAR}_{END_YEAR}.csv", index=False)
by_race_all.to_csv(f"pairwise_accuracy_by_race_{START_YEAR}_{END_YEAR}.csv", index=False)
by_driver_all.to_csv(f"pairwise_accuracy_by_driver_{START_YEAR}_{END_YEAR}.csv", index=False)

2018: races=21, pairs=3990, accuracy=0.7115
2019: races=21, pairs=3990, accuracy=0.6967
2020: races=17, pairs=3230, accuracy=0.6858
2021: races=22, pairs=4180, accuracy=0.7306
2022: races=22, pairs=4180, accuracy=0.6921

=== Overall Accuracy by Year ===
   year  num_races  num_pairs  num_correct  accuracy
0  2018         21       3990         2839  0.711529
1  2019         21       3990         2780  0.696742
2  2020         17       3230         2215  0.685759
3  2021         22       4180         3054  0.730622
4  2022         22       4180         2893  0.692105

=== Overall Accuracy for Full Range ===
   start_year  end_year  num_years  num_races  num_pairs  num_correct  \
0        2018      2022          5        103      19570        13781   

   accuracy  
0   0.70419  

=== Best Races by Accuracy ===
    year  raceId                raceName  circuitId  num_pairs  num_correct  \
0   2018    1003    Singapore Grand Prix         15        190          161   
21  2019    1017      

# Prepare Submission

To get full credit for this assignment, you should submit your assignment in two formats so that we can easily grade and debug your code:
1. **.ipynb:** First, confirm that your code can run from start to finish without any errors. To check this, go to "Runtime" > "Run all" in the Google Colab menu. If everything looks good, you can export your file by going to "File" > "Download" > "Download .ipynb".
2. **.pdf:** Run the function called `colab2pdf()` below. This will automatically convert your notebook to a PDF. Note that while "File" > "Print" > "Save as PDF" also works, it requires you to manually expand all of the cells and may cut off some images.

In [24]:
colab2pdf()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>